
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png" alt="Databricks Learning">
</div>


# Lab - Model Deployment with Spark 

In this lab, you will gain hands-on experience in deploying machine learning models using Apache Spark and optimizing query performance with Delta Lake. You will explore both single-node and distributed model deployment strategies, log models using MLflow for tracking, and apply advanced Delta Lake optimization techniques like **OPTIMIZE**, **VACUUM**, and **Liquid Clustering** to enhance performance and resource efficiency.

**Lab Outline:**

By the end of this lab, you will be able to:

- **Task 1: Single-Node Model Deployment**  
  - Train a single-node **Gradient Boosted Tree (GBT)** model using **Scikit-learn**.
  - Log the trained model in **MLflow**.
  - Use **Spark UDFs** to perform parallelized inference on distributed data.

- **Task 2: Distributed Model Deployment with Spark MLlib**  
  - Train a distributed **Gradient Boosted Tree (GBT)** model using Spark MLlib.
  - Log the model using **MLflow** and set a model alias for easy reference.
  - Perform distributed inference and save predictions to a Delta table.

- **Task 3: Delta Lake Optimizations**  
  - Apply Delta Lake optimization strategies like **OPTIMIZE** to compact small files.
  - Perform **VACUUM** to clean up old, unused data files.
  - Apply **Z-ORDER Clustering** for faster reads on frequently queried columns.
  - Enable **Liquid Clustering** to incrementally optimize data layout.

- **Task 4: Performance Comparison Before and After Optimization**  
  - Measure query performance before and after applying Delta Lake optimizations.
  - Compare the improvements in query speed and resource efficiency.

## REQUIRED - SELECT CLASSIC COMPUTE
Before executing cells in this notebook, please select your classic compute cluster in the lab. Be aware that **Serverless** is enabled by default.

Follow these steps to select the classic compute cluster:
1. Navigate to the top-right of this notebook and click the drop-down menu to select your cluster. By default, the notebook will use **Serverless**.

2. If your cluster is available, select it and continue to the next cell. If the cluster is not shown:

   - Click **More** in the drop-down.
   
   - In the **Attach to an existing compute resource** window, use the first drop-down to select your unique cluster.

**NOTE:** If your cluster has terminated, you might need to restart it in order to select it. To do this:

1. Right-click on **Compute** in the left navigation pane and select *Open in new tab*.

2. Find the triangle icon to the right of your compute cluster name and click it.

3. Wait a few minutes for the cluster to start.

4. Once the cluster is running, complete the steps above to select your cluster.

## Requirements

Please review the following requirements before starting the lesson:

* To run this notebook, you need a classic cluster running one of the following Databricks runtime(s): **16.3.x-cpu-ml-scala2.12**. **Do NOT use serverless compute to run this notebook**.

## Classroom Setup

Install required libraries.

In [0]:
%pip install -U optuna mlflow==2.9.2 delta-spark joblibspark pyspark==3.5.3

dbutils.library.restartPython()

  Using cached optuna-4.5.0-py3-none-any.whl.metadata (17 kB)
  Using cached mlflow-2.9.2-py3-none-any.whl.metadata (13 kB)
  Using cached delta_spark-4.0.0-py3-none-any.whl.metadata (1.9 kB)
  Using cached joblibspark-0.6.0-py3-none-any.whl.metadata (1.2 kB)
  Using cached pyspark-3.5.3-py2.py3-none-any.whl
  Using cached databricks_cli-0.18.0-py2.py3-none-any.whl.metadata (4.0 kB)
  Using cached pytz-2023.4-py2.py3-none-any.whl.metadata (22 kB)
  Using cached packaging-23.2-py3-none-any.whl.metadata (3.2 kB)
  Using cached docker-6.1.3-py3-none-any.whl.metadata (3.5 kB)
  Using cached querystring_parser-1.2.4-py2.py3-none-any.whl.metadata (559 bytes)
  Using cached pyarrow-14.0.2-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (3.0 kB)
  Using cached py4j-0.10.9.7-py2.py3-none-any.whl.metadata (1.5 kB)
INFO: pip is looking at multiple versions of delta-spark to determine which version is compatible with other requirements. This could take a while.
  Using cached delta_spark-3.3.2-py3-

Before starting the demo, run the provided classroom setup script.

In [0]:
%run "../Includes/Classroom-Setup-lab"

  Using cached databricks_sdk-0.36.0-py3-none-any.whl.metadata (38 kB)
Using cached databricks_sdk-0.36.0-py3-none-any.whl (569 kB)
  Attempting uninstall: databricks-sdk
    Found existing installation: databricks-sdk 0.30.0
    Not uninstalling databricks-sdk at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-63662835-36db-44fc-9396-880afc1c02f5
    Can't uninstall 'databricks-sdk'. No files were found to uninstall.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


Created large Delta table at /Volumes/dbacademy/ops/labuser11731658_1758653716@vocareum_com/v01/large_california_housing_delta


**Other Conventions:**

Throughout this demo, we'll refer to the object `DA`. This object, provided by Databricks Academy, contains variables such as your username, catalog name, schema name, working directory, and dataset locations. Run the code block below to view these details:

In [0]:
print(f"Username:          {DA.username}")
print(f"Catalog Name:      {DA.catalog_name}")
print(f"Schema Name:       {DA.schema_name}")
print(f"Working Directory: {DA.paths.working_dir}")
print(f"Dataset Location:  {DA.paths.datasets.california_housing}")

Username:          labuser11731658_1758653716@vocareum.com
Catalog Name:      dbacademy
Schema Name:       labuser11731658_1758653716
Working Directory: /Volumes/dbacademy/ops/labuser11731658_1758653716@vocareum_com
Dataset Location:  /Volumes/dbacademy_california_housing/v01


##Pre-Steps: Data Preparation and Feature Engineering
Before you dive into model deployment, you need to prepare the California Housing dataset for model training and testing. This includes loading the dataset from Delta format, performing feature engineering, and splitting the data into training and testing sets.


###Loading the California Housing Dataset
In this step, you'll load the California Housing dataset from Delta Lake and prepare it for both Spark and Scikit-learn model training. The dataset will be split into an 80/20 ratio for training and testing.

In [0]:
from sklearn.model_selection import train_test_split
from pyspark.ml.feature import VectorAssembler

# Load the California Housing dataset from Delta table
data_path = f"{DA.paths.datasets.california_housing}/data"
df = spark.read.format("delta").load(data_path)

# Define feature columns
feature_columns = [
    "MedInc", "HouseAge", "AveRooms", "AveBedrms", 
    "Population", "AveOccup", "Latitude", "Longitude"
]

# Assemble features into a vector
assembler = VectorAssembler(inputCols=feature_columns, outputCol="features")
df_with_features = assembler.transform(df)

# Split the data into training and test sets (for SparkML Model Training)
train_df, test_df = df_with_features.randomSplit([0.8, 0.2], seed=42)

# Convert Spark DataFrame to Pandas DataFrame for single-node model training
train_pandas_df = train_df.select(feature_columns + ["label"]).toPandas()
test_pandas_df = test_df.select(feature_columns + ["label"]).toPandas()

# Split features and target variable for Scikit-learn training
X_train = train_pandas_df.drop(columns=["label"])
y_train = train_pandas_df["label"]
X_test = test_pandas_df.drop(columns=["label"])
y_test = test_pandas_df["label"]

# Display the first few rows of the dataset to verify data loading
display(df.limit(10))

display(df_with_features.limit(10))

X_train.head(10)

MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,label
0.4999,16.0,21.63157894736842,6.0,26.0,1.368421052631579,39.42,-122.89,0.735
0.4999,23.0,6.054545454545455,1.6727272727272726,198.0,3.6,36.09,-119.99,1.0
0.4999,46.0,1.7142857142857142,0.5714285714285714,18.0,2.5714285714285716,37.81,-122.29,0.675
0.536,26.0,7.846153846153846,1.3076923076923077,43.0,3.3076923076923075,38.7,-122.52,0.875
0.536,46.0,3.142857142857143,1.0476190476190477,37.0,1.7619047619047619,38.02,-121.84,0.875
0.5495,38.0,4.249056603773585,1.0188679245283019,999.0,3.769811320754717,33.96,-118.27,0.917
0.716,39.0,4.730769230769231,1.0961538461538463,316.0,6.076923076923077,37.96,-122.36,1.042
0.7286,46.0,3.375451263537906,1.0722021660649819,582.0,2.101083032490975,37.81,-122.29,0.952
0.7403,37.0,4.491428571428571,1.1485714285714286,1046.0,2.9885714285714284,37.96,-122.37,0.686
0.7683,38.0,4.253561253561253,1.0541310541310542,1144.0,3.259259259259259,37.76,-122.19,0.818


MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,label,features
0.4999,16.0,21.63157894736842,6.0,26.0,1.368421052631579,39.42,-122.89,0.735,"Map(vectorType -> dense, length -> 8, values -> List(0.4999, 16.0, 21.63157894736842, 6.0, 26.0, 1.368421052631579, 39.42, -122.89))"
0.4999,23.0,6.054545454545455,1.6727272727272726,198.0,3.6,36.09,-119.99,1.0,"Map(vectorType -> dense, length -> 8, values -> List(0.4999, 23.0, 6.054545454545455, 1.6727272727272726, 198.0, 3.6, 36.09, -119.99))"
0.4999,46.0,1.7142857142857142,0.5714285714285714,18.0,2.5714285714285716,37.81,-122.29,0.675,"Map(vectorType -> dense, length -> 8, values -> List(0.4999, 46.0, 1.7142857142857142, 0.5714285714285714, 18.0, 2.5714285714285716, 37.81, -122.29))"
0.536,26.0,7.846153846153846,1.3076923076923077,43.0,3.3076923076923075,38.7,-122.52,0.875,"Map(vectorType -> dense, length -> 8, values -> List(0.536, 26.0, 7.846153846153846, 1.3076923076923077, 43.0, 3.3076923076923075, 38.7, -122.52))"
0.536,46.0,3.142857142857143,1.0476190476190477,37.0,1.7619047619047619,38.02,-121.84,0.875,"Map(vectorType -> dense, length -> 8, values -> List(0.536, 46.0, 3.142857142857143, 1.0476190476190477, 37.0, 1.7619047619047619, 38.02, -121.84))"
0.5495,38.0,4.249056603773585,1.0188679245283019,999.0,3.769811320754717,33.96,-118.27,0.917,"Map(vectorType -> dense, length -> 8, values -> List(0.5495, 38.0, 4.249056603773585, 1.0188679245283019, 999.0, 3.769811320754717, 33.96, -118.27))"
0.716,39.0,4.730769230769231,1.0961538461538463,316.0,6.076923076923077,37.96,-122.36,1.042,"Map(vectorType -> dense, length -> 8, values -> List(0.716, 39.0, 4.730769230769231, 1.0961538461538463, 316.0, 6.076923076923077, 37.96, -122.36))"
0.7286,46.0,3.375451263537906,1.0722021660649819,582.0,2.101083032490975,37.81,-122.29,0.952,"Map(vectorType -> dense, length -> 8, values -> List(0.7286, 46.0, 3.375451263537906, 1.0722021660649819, 582.0, 2.101083032490975, 37.81, -122.29))"
0.7403,37.0,4.491428571428571,1.1485714285714286,1046.0,2.9885714285714284,37.96,-122.37,0.686,"Map(vectorType -> dense, length -> 8, values -> List(0.7403, 37.0, 4.491428571428571, 1.1485714285714286, 1046.0, 2.9885714285714284, 37.96, -122.37))"
0.7683,38.0,4.253561253561253,1.0541310541310542,1144.0,3.259259259259259,37.76,-122.19,0.818,"Map(vectorType -> dense, length -> 8, values -> List(0.7683, 38.0, 4.253561253561253, 1.0541310541310542, 1144.0, 3.259259259259259, 37.76, -122.19))"


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,0.4999,43.0,7.846154,1.461538,44.0,3.384615,38.07,-120.19
1,0.5360,16.0,2.111111,2.111111,166.0,18.444444,37.67,-121.04
2,0.5360,36.0,12.250000,3.500000,18.0,2.250000,40.31,-123.17
3,0.6960,52.0,5.333333,1.592593,272.0,5.037037,37.95,-121.29
4,0.7054,43.0,2.229167,0.916667,107.0,2.229167,33.12,-117.08
5,0.7445,19.0,3.567568,1.045045,641.0,2.887387,36.33,-119.29
6,0.7800,10.0,3.835766,1.083942,927.0,3.383212,36.33,-119.28
7,0.7917,52.0,2.018868,1.490566,167.0,3.150943,37.95,-121.29
8,0.8106,16.0,3.430000,1.085000,614.0,3.070000,34.64,-120.46
9,0.8130,12.0,4.781095,1.701493,315.0,1.567164,33.12,-117.10


## Task 1: Single-Node Model Deployment with XGBoost and Spark UDFs

In this task, you will train an **XGBRegressor** model using a **single-node training** approach. After training the model, you will log it using **MLflow** to enable tracking and version control. Additionally, you will apply **Spark UDFs** to perform parallelized inference on distributed data using the trained model. This approach demonstrates how to deploy a single-node model and leverage Spark's distributed computing capabilities for inference.

**Instructions:**

1. **Train an XGBRegressor model** on the provided training data.
2. **Log the trained model** in MLflow for tracking and reproducibility.
3. **Perform inference** on the test data to evaluate model predictions.
4. **Apply Spark UDFs** to deploy the model and perform distributed, parallelized inference on the cluster.

### Task 1.1: Training and Logging the XGBoost Model  

In this step, you will train an **XGBRegressor** model on the training dataset. After training, you will log the model in MLflow, allowing you to track the model's parameters and performance metrics for future reference.

**Steps:**
- **Step 1:** Train an **XGBRegressor** model with specified parameters on the training data.
- **Step 2:** Log the trained model using MLflow to track the training run, including metrics like training and inference times.

In [0]:
import xgboost as xgb
from xgboost import XGBRegressor
import mlflow
import mlflow.xgboost
import time
# Set the active experiment
mlflow.set_experiment(f"/Users/{DA.username}/lab_experiment")
# Train the XGBRegressor model
xgb_model = XGBRegressor(
    objective="reg:squarederror",
    max_depth=5,
    learning_rate=0.1,
    n_estimators=100,
    random_state=42
)

# Measure training time
start_time = time.time()
xgb_model.fit(X_train, y_train)

training_time_single_node = time.time() - start_time
print(f"Training time (single-node): {training_time_single_node:.2f} seconds")

# Log the XGBRegressor model with MLflow
with mlflow.start_run() as run:
    mlflow.xgboost.log_model(xgb_model, "xgboost_model_single_node")
    run_id_single_node = run.info.run_id  # Capture the MLflow run ID for later use
    print(f"Model logged in run: {run_id_single_node}")

    # Measure inference time
    start_time = time.time()
    y_pred = xgb_model.predict(X_test)
    inference_time_single_node = time.time() - start_time
    print(f"Inference time (single-node): {inference_time_single_node:.2f} seconds")

Training time (single-node): 0.55 seconds


2025/09/23 21:12:31 WARNING mlflow.models.model: Model logged without a signature. Signatures will be required for upcoming model registry features as they validate model inputs and denote the expected schema of model outputs. Please visit https://www.mlflow.org/docs/2.9.2/models.html#set-signature-on-logged-model for instructions on setting a model signature on your logged model.


Uploading artifacts:   0%|          | 0/5 [00:00<?, ?it/s]

Model logged in run: 12c74b981da3463b90d4e836e8d2ab0d
Inference time (single-node): 0.02 seconds


### Task 1.2: Performing Parallelized Inference Using Spark UDFs  

In this step, you will use Spark UDFs to perform distributed inference by applying the trained **XGBRegressor** model across a Spark DataFrame. This approach allows you to scale the inference process across the cluster, taking advantage of Spark's parallel processing capabilities.

**Steps:**
- **Step 1:** Load the trained XGBRegressor model from MLflow using its URI.
- **Step 2:** Define a Spark UDF that applies the loaded model to perform distributed inference on each data partition.
- **Step 3:** Display the results, including the original features, true labels, and predicted values, to evaluate the model's predictions on distributed data.


In [0]:
from pyspark.sql.functions import pandas_udf, struct
import mlflow.pyfunc

# Perform inference on the test set
y_pred = xgb_model.predict(X_test)
# Load the logged model from MLflow
model_uri = f"runs:/{run_id_single_node}/xgboost_model_single_node"
predict_udf = mlflow.pyfunc.spark_udf(spark, model_uri, result_type="double")

# Apply the UDF for inference on the distributed dataset
predictions = df_with_features.withColumn("predicted_label", predict_udf(struct(*feature_columns)))

# Display the predictions along with original features and labels
display(predictions.select(feature_columns + ["label", "predicted_label"]))

2025/09/23 21:15:46 WARNING mlflow.pyfunc: Calling `spark_udf()` with `env_manager="local"` does not recreate the same environment that was used during training, which may lead to errors or inaccurate predictions. We recommend specifying `env_manager="conda"`, which automatically recreates the environment that was used to train the model and performs inference in the recreated environment.


2025/09/23 21:15:47 INFO mlflow.models.flavor_backend_registry: Selected backend for flavor 'python_function'


MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,label,predicted_label
0.536,26.0,6.0,1.0,5.0,1.6666666666666667,33.62,-114.62,2.75,0.7954514622688293
0.7069,29.0,3.0934579439252334,0.794392523364486,341.0,3.1869158878504673,34.13,-117.34,0.703,0.882117509841919
0.7075,35.0,3.2908496732026142,1.1062091503267975,1714.0,2.8006535947712417,34.11,-117.29,0.788,0.99688720703125
0.7526,5.0,2.5796610169491525,1.0372881355932204,2031.0,6.884745762711864,38.58,-121.5,1.625,1.8410650491714478
0.8691,18.0,4.036764705882353,1.0514705882352942,249.0,1.8308823529411764,38.78,-121.24,1.365,1.1901956796646118
0.8926,10.0,6.0,1.1008403361344539,381.0,3.2016806722689077,34.14,-117.46,1.161,0.8702031970024109
0.8991,42.0,4.689922480620155,1.1317829457364341,945.0,3.6627906976744184,32.7,-117.13,0.789,0.8813141584396362
0.9142,52.0,2.0153452685421995,1.184143222506394,805.0,2.0588235294117645,32.72,-117.16,1.625,1.9732869863510132
0.9196,52.0,1.550408719346049,1.103542234332425,509.0,1.3869209809264305,38.58,-121.49,1.375,1.520622730255127
0.9274,38.0,4.3428571428571425,0.8571428571428571,65.0,1.8571428571428572,38.53,-121.4,0.675,0.7481151819229126


## Task 2: Distributed Model Deployment with Spark MLlib  

In this task, you will train and deploy a **XGBRegressor** in a distributed environment using **Spark MLlib**. You will log the trained model with **MLflow** and perform distributed inference on the test dataset. Additionally, the predictions will be saved to a Delta table for further analysis. 

**Instructions:**

1. **Train a distributed XGBRegressor model** using Spark MLlib.
2. **Log the trained model** in MLflow for future use.
3. **Perform distributed inference** on the test dataset using the trained model.
4. **Save predictions** to a Delta table in Unity Catalog.

### Task 2.1: Training and Logging the XGBRegressor Model

In this task, you will train an **XGBRegressor** model using the distributed Spark DataFrame. By preparing the features within Spark and converting to Pandas, you will leverage the power of both Spark for data processing and XGBoost for model training. After training, you will log the model in MLflow for version control and future use.

**Steps:**
- **Step 1:** Prepare the features using **VectorAssembler** to transform the data for model training.
- **Step 2:** Convert the training data into a Pandas DataFrame and train the **XGBRegressor** model.
- **Step 3:** Log the trained model in **MLflow**, including its signature, to enable model tracking and reproducibility.
- **Step 4:** Measure and print both the training and inference times to assess model performance.
- **Step 5:** Save the trained model's URI in MLflow for easy reference in subsequent tasks.

In [0]:
# Distributed Model Deployment with XGBRegressor using Spark MLlib
from pyspark.ml.feature import VectorAssembler
from pyspark.ml import Pipeline
from pyspark.sql.functions import struct
import mlflow
from mlflow.tracking import MlflowClient
from mlflow.models import infer_signature
import xgboost as xgb
from xgboost import XGBRegressor
import time
import pandas as pd

# Set the MLflow registry URI to Unity Catalog
mlflow.set_registry_uri("databricks-uc")

# Create a VectorAssembler to prepare the features
assembler = VectorAssembler(inputCols=feature_columns, outputCol="features")

# Assemble the features in Spark DataFrame
df_with_features = assembler.transform(df)

# Split the data into training and test sets
train_df, test_df = df_with_features.randomSplit([0.8, 0.2], seed=42)

# Prepare the training data as Pandas DataFrame for XGBoost
X_train = train_df.select(feature_columns).toPandas()
y_train = train_df.select("label").toPandas()
X_test = test_df.select(feature_columns).toPandas()
y_test = test_df.select("label").toPandas()

# Create the XGBRegressor model
xgb_regressor = XGBRegressor(
    objective="reg:squarederror",
    max_depth=5,
    learning_rate=0.1,
    n_estimators=100,
    random_state=42
)

# Measure training time
start_time = time.time()
xgb_regressor.fit(X_train, y_train)
training_time_distributed = time.time() - start_time
print(f"Training time (distributed): {training_time_distributed} seconds")

# Log the XGBRegressor model with MLflow
with mlflow.start_run() as run:
    # Log the XGBRegressor model and infer its signature
    signature = infer_signature(X_train, xgb_regressor.predict(X_train))
    mlflow.xgboost.log_model(
        xgb_model=xgb_regressor,
        artifact_path="xgboost_model", 
        signature=signature)
    run_id_distributed = run.info.run_id

# Measure inference time
start_time = time.time()
y_pred = xgb_regressor.predict(X_test)
inference_time_distributed = time.time() - start_time
print(f"Inference time (distributed): {inference_time_distributed} seconds")

# Save the trained model's URI for inference
model_uri = f"runs:/{run_id_distributed}/xgboost_model"
print(f"Model saved at {model_uri}")

Training time (distributed): 0.28895092010498047 seconds


Uploading artifacts:   0%|          | 0/5 [00:00<?, ?it/s]

Inference time (distributed): 0.003861665725708008 seconds
Model saved at runs:/4649913d9c6c41e7a48c9df5094b5e49/xgboost_model


### Task 2.2: Performing Distributed Inference and Saving Predictions to Delta Table

In this step, you will apply the trained **Gradient Boosted Tree (GBT)** model to the test dataset and perform distributed inference across the Spark cluster. The predictions will be saved to a Delta table in Unity Catalog for further analysis.

**Steps:**
- **Step 1:** Convert the test dataset to a Pandas DataFrame and use the trained **XGBRegressor** model to perform inference.
- **Step 2:** Combine the original test dataset with the predicted values and convert the results back to a Spark DataFrame.
- **Step 3:** Save the predictions to a Delta table for future analysis.
- **Step 4:** Explore the results by displaying the predictions alongside the actual labels, allowing for a comparison of the model's performance.

In [0]:
# Perform inference on the test dataset using the distributed XGBRegressor model
import pandas as pd
from pyspark.sql import functions as F

# Convert the test dataset to a Pandas DataFrame for prediction
X_test = test_df.select(feature_columns).toPandas()

# Perform inference using the trained XGBRegressor model
y_pred = xgb_regressor.predict(X_test)

# Convert predictions to a Pandas DataFrame
predictions_pd = pd.DataFrame(y_pred, columns=['predicted_label'])

# Combine the original test DataFrame with the predictions
test_df_with_preds = test_df.toPandas()  # Convert test_df to Pandas DataFrame
test_df_with_preds["predicted_label"] = predictions_pd["predicted_label"]

# Convert the combined DataFrame back to a Spark DataFrame
predictions_spark_df =spark.createDataFrame(test_df_with_preds)

# Display predictions
display(predictions_spark_df.select(*feature_columns, "label", "predicted_label"))

# Save predictions to a Delta table in Unity Catalog
predictions_spark_df.write.format("delta").mode("overwrite").saveAsTable(f"{DA.catalog_name}.{DA.schema_name}.distributed_xgb_predictions_table")

MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,label,predicted_label
0.536,33.0,4.928571428571429,2.0,47.0,3.357142857142857,37.44,-121.31,1.125,1.5116337537765503
0.7068,49.0,2.0822784810126582,1.0063291139240507,467.0,1.4778481012658229,37.33,-121.89,2.0,2.712958812713623
0.7714,16.0,2.698581560283688,1.0851063829787233,438.0,1.553191489361702,37.95,-121.29,0.875,1.5604547262191772
0.8229,30.0,3.763819095477387,1.0753768844221105,537.0,2.698492462311558,36.21,-119.34,0.684,0.7744726538658142
0.9336,10.0,3.6627450980392156,1.0196078431372548,401.0,1.572549019607843,37.66,-120.98,1.271,1.9386814832687378
0.9573,19.0,2.951219512195122,1.0,658.0,16.048780487804876,37.64,-121.0,1.625,1.0373815298080444
1.077,17.0,3.0548712206047033,1.0167973124300111,1670.0,1.87010078387458,37.78,-122.43,1.15,2.656965494155884
1.125,31.0,3.7004160887656035,1.1262135922330097,1398.0,1.9389736477115118,37.96,-121.3,1.104,1.2198618650436401
1.2387,42.0,4.248741188318228,1.1329305135951662,3549.0,3.5740181268882174,37.72,-122.42,2.128,1.8225936889648438
1.24,24.0,2.967984934086629,1.0922787193973635,1630.0,3.0696798493408664,34.41,-119.86,3.25,2.2514076232910156


**Further Exploration:**  

For more information on model tuning and deployment using Spark MLlib, refer to the [Spark MLlib Guide](https://spark.apache.org/docs/latest/ml-guide.html).

## Task 3: Delta Lake Optimization Strategies

In this section, you will explore how to optimize query performance and storage efficiency using Delta Lake features like **OPTIMIZE**, **VACUUM**, and **Liquid Clustering**. These optimization techniques will help compact data files, clean up unused files, and enhance query performance, especially for frequently accessed data.

**Key Delta Lake Optimization Features**:
- **OPTIMIZE**: Compacts small files into larger ones, improving query read performance.
- **VACUUM**: Removes old, unused data files, optimizing storage and ensuring a clean data state.
- **Z-ORDER Clustering**: Clusters data based on columns frequently used in queries, speeding up read times by organizing the data more efficiently.
- **Liquid Clustering**: Automatically adjusts clustering over time based on query patterns for optimal performance.

**Task Outline:**

- **Step 1:** Apply **OPTIMIZE** to compact small files in the Delta table.
- **Step 2:** Apply **VACUUM** to clean up old files and reclaim storage.
- **Step 3:** Enable **Liquid Clustering** for automatic optimization.
- **Step 4:** Measure query performance before and after optimization.

#### Enable Predictive Optimization for Your Account
You must enable predictive optimization at the account level. You can then enable or disable predictive optimization at the catalog and schema levels.

An account admin must complete the following steps to enable predictive optimization for all metastores in an account:

* **Step 1:** Access the [Accounts Console](https://accounts.cloud.databricks.com/login).

* **Step 2:** Navigate to **Settings**, then **Feature enablement**.

* **Step 3:** Select **Enabled** next to **Predictive optimization**.

### Task 3.1: Applying OPTIMIZE to the Delta Table  

In this task, you will apply **OPTIMIZE** on the Delta table to compact small files into larger ones. This operation reduces the overhead of reading many small files during queries, improving the overall performance of data reads.

**Steps:**
- **Step 1:** Run the `OPTIMIZE` command on your Delta table.

In [0]:
# OPTIMIZE the Delta table to compact small files
spark.sql(f"""
OPTIMIZE {DA.catalog_name}.{DA.schema_name}.distributed_xgb_predictions_table
""") 

DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,numOutputZCubes:bigint>,numBins:bigint,numBatches:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,preserveInsertionOrder:boolean,numFilesSkippedToReduceWriteAmplification:bigint,numBytesSkippedToReduceWriteAmplification:bigint,startTimeMs:bigint,endTimeMs:bigint,

### Task 3.2: Applying VACUUM to Clean Up Old Files

The **VACUUM** operation removes old, unused data files from the Delta table that are no longer referenced by the latest version of the table. This helps reclaim storage space and ensures that only the necessary data files are retained.

**Step:**  
- **Step 1:** Run the VACUUM operation to remove old, unused data files from your Delta table.

In [0]:
# Disable the retention duration check and apply VACUUM
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")

# Apply VACUUM to clean up old files and reclaim storage space
display(spark.sql(f"""
VACUUM {DA.catalog_name}.{DA.schema_name}.distributed_xgb_predictions_table RETAIN 0 HOURS
"""))

path
s3://unity-catalogs-us-west-2/metastore/3875221-root/47405c70-0f83-4b4c-a9b5-f2c6559736ec/tables/78588530-6718-4afb-8a3d-a95717687a4a


For more information on distributed machine learning with Spark, visit [Spark MLlib's Guide](https://spark.apache.org/docs/latest/ml-guide.html).

### Task 3.3: Enabling Liquid Clustering

**Liquid Clustering** is an advanced Delta Lake feature that dynamically clusters your data based on the query patterns it observes over time. This ensures that frequently queried columns are automatically clustered, leading to faster reads.

**Steps:**  
- **Step 1:** Enable **Liquid Clustering** on your Delta table by clustering it based on frequently queried columns.
- **Step 2:** Query the Delta table's history to check the clustering progress after enabling Liquid Clustering.

In [0]:
%sql
-- Enable Liquid Clustering on the Delta table, clustering by frequently queried columns
ALTER TABLE distributed_xgb_predictions_table
CLUSTER BY (label);


-- Display the improvement after applying liquid clustering
DESCRIBE HISTORY distributed_xgb_predictions_table

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
4,2025-09-23T21:36:06Z,76230592971519,labuser11731658_1758653716@vocareum.com,CLUSTER BY,"Map(oldClusteringColumns -> , newClusteringColumns -> label)",null,List(1903700905083956),0923-185603-ggr7be56,3,WriteSerializable,true,Map(),null,Databricks-Runtime/16.3.x-cpu-ml-scala2.12
3,2025-09-23T21:36:05Z,76230592971519,labuser11731658_1758653716@vocareum.com,ROW TRACKING BACKFILL,Map(batchId -> 0),null,List(1903700905083956),0923-185603-ggr7be56,2,SnapshotIsolation,false,Map(),null,Databricks-Runtime/16.3.x-cpu-ml-scala2.12
2,2025-09-23T21:36:04Z,76230592971519,labuser11731658_1758653716@vocareum.com,UPGRADE PROTOCOL,"Map(newProtocol -> {""minReaderVersion"":3,""minWriterVersion"":7,""readerFeatures"":[""deletionVectors""],""writerFeatures"":[""deletionVectors"",""domainMetadata"",""rowTracking"",""invariants"",""appendOnly""]})",null,List(1903700905083956),0923-185603-ggr7be56,1,WriteSerializable,true,Map(),null,Databricks-Runtime/16.3.x-cpu-ml-scala2.12
1,2025-09-23T21:32:19Z,76230592971519,labuser11731658_1758653716@vocareum.com,OPTIMIZE,"Map(predicate -> [], auto -> false, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(1903700905083956),0923-185603-ggr7be56,0,SnapshotIsolation,false,"Map(numRemovedFiles -> 4, numRemovedBytes -> 299585, p25FileSize -> 273066, numDeletionVectorsRemoved -> 0, minFileSize -> 273066, numAddedFiles -> 1, maxFileSize -> 273066, p75FileSize -> 273066, p50FileSize -> 273066, numAddedBytes -> 273066)",null,Databricks-Runtime/16.3.x-cpu-ml-scala2.12
0,2025-09-23T21:29:16Z,76230592971519,labuser11731658_1758653716@vocareum.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.enableDeletionVectors"":""true""}, statsOnLoad -> false)",null,List(1903700905083956),0923-185603-ggr7be56,null,WriteSerializable,false,"Map(numFiles -> 4, numRemovedFiles -> 0, numRemovedBytes -> 0, numOutputRows -> 2818, numOutputBytes -> 299585)",null,Databricks-Runtime/16.3.x-cpu-ml-scala2.12


### Task 3.4: Measuring Query Performance Before and After Optimization  

To truly understand the impact of these optimizations, we will measure the query performance before and after applying the Delta Lake optimizations. This will help you observe the improvements in query speed and resource efficiency.

**Steps:**  
- **Step 1:** Run the same query on your Delta table before and after applying the optimizations.
- **Step 2:** Measure the time taken for the query before optimization.
- **Step 3:** Apply **OPTIMIZE** and **VACUUM**.
- **Step 4:** Measure the query performance again and compare the improvement in speed.
- **Step 5:** Display the performance improvement percentage and discuss how the Delta Lake optimizations affected the query time.

In [0]:
from time import time

# Define the table and query
table_name = f"{DA.catalog_name}.{DA.schema_name}.distributed_xgb_predictions_table"

# Define the query you will run before and after optimizations
query = f"""
SELECT label, COUNT(*)
FROM {table_name}
GROUP BY label
"""

# Measure query performance before optimization
start_time_before = time()
df_before = spark.sql(query)
time_before = time() - start_time_before
print(f"Query Execution Time Before Optimization: {time_before:.2f} seconds")

# Apply Delta Lake optimizations (OPTIMIZE and VACUUM were already applied earlier)

# Measure query performance after optimization
start_time_after = time()
df_after = spark.sql(query)
time_after = time() - start_time_after
print(f"Query Execution Time After Optimization: {time_after:.2f} seconds")

# Calculate the performance improvement
performance_improvement = (time_before - time_after) / time_before * 100
print(f"Performance improvement after optimizations: {performance_improvement:.2f}%")

Query Execution Time Before Optimization: 0.14 seconds
Query Execution Time After Optimization: 0.08 seconds
Performance improvement after optimizations: 42.06%


## Conclusion

In this lab, you successfully explored various methods for deploying machine learning models using Spark, including single-node and distributed deployment strategies. You gained hands-on experience with:

- Preparing data for model training and deployment using the Wine Quality dataset.
- Training and deploying a single-node machine learning model using Scikit-learn, followed by leveraging **Spark UDFs** for parallelized inference on distributed data.
- Training and deploying a distributed machine learning model using **Spark MLlib**, followed by performing distributed inference and logging the model with MLflow.
- Applying **Delta Lake optimizations** such as **OPTIMIZE**, **VACUUM**, and **Liquid Clustering** to enhance query performance, reduce storage costs, and improve overall resource efficiency.

Finally, you compared the query performance before and after these optimizations, observing how Delta Lake's features significantly improved the speed and efficiency of your model serving pipeline.



&copy; 2025 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="blank">Apache Software Foundation</a>.<br/>
<br/><a href="https://databricks.com/privacy-policy" target="blank">Privacy Policy</a> | 
<a href="https://databricks.com/terms-of-use" target="blank">Terms of Use</a> | 
<a href="https://help.databricks.com/" target="blank">Support</a>